# Unidad 1 del curso de Reinforcement Learning de Hugging Face
Lunar lander
# Dependencias

In [3]:
import gymnasium as gym

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

# Environment
Vamos a usar Lunar lander v3 pero hay distintas formas de visualizarlo

In [ ]:
# La tipica:
env = gym.make("LunarLander-v3")
observation, info = env.reset()
print(f"Action space: {env.action_space.n}")
print(f"Observation space: {env.observation_space.sample()}")

# Tambien se puede obtener una forma vectorizada
env = make_vec_env('LunarLander-v3', n_envs=16)

Action space: 4
Observation space: [-2.2544854  -1.5142559  -6.264372    1.3950449   6.057291    4.0967846
  0.33680052  0.3002461 ]
Action space: 4
Observation space: [ 0.27378187  0.18526086  2.7005434  -9.298824    1.0136071  -2.077992
  0.91144943  0.6002873 ]


## Modelo
Vamos a usar PPO y como las caracteristicas van a ser vectoriales, la policy será una MLP


In [17]:
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

Using cpu device


Lo vamos a entrenar durante 500_000 timesteps

In [ ]:
model.learn(total_timesteps=300_000)

# Y lo guardamos:
model_name = "ppo-LunarLander-v3"
model.save(model_name)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 111      |
|    ep_rew_mean     | -51.3    |
| time/              |          |
|    fps             | 4634     |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 121         |
|    ep_rew_mean          | -34.9       |
| time/                   |             |
|    fps                  | 2534        |
|    iterations           | 2           |
|    time_elapsed         | 12          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.010566788 |
|    clip_fraction        | 0.0768      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.25       |
|    explained_variance   | -1.55e-06   |
|    learning_rate        | 0.

KeyboardInterrupt: 

Evaluar al agente

In [23]:
import gymnasium as gym
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium.wrappers import RecordVideo

env = gym.make("LunarLander-v3", render_mode="rgb_array")
env = RecordVideo(env, video_folder="./videos", episode_trigger=lambda x: True)
eval_env = Monitor(env)

mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True
)

print(f"mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

eval_env.close()


c:\repos\RL_HF\.venv\Lib\site-packages\gymnasium\wrappers\rendering.py:292: UserWarning: WARN: Overwriting existing videos at c:\repos\RL_HF\notebooks\Unit_1\videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


mean_reward=109.56 +/- 119.78


In [24]:
import glob
from IPython.display import Video, display

videos = sorted(glob.glob("./videos/*.mp4"))

for video in videos:
    print(video)
    display(Video(video, embed=True))

./videos\rl-video-episode-0.mp4


./videos\rl-video-episode-1.mp4


./videos\rl-video-episode-10.mp4


./videos\rl-video-episode-2.mp4


./videos\rl-video-episode-3.mp4


...

./videos\rl-video-episode-5.mp4


./videos\rl-video-episode-6.mp4


./videos\rl-video-episode-7.mp4


./videos\rl-video-episode-8.mp4


./videos\rl-video-episode-9.mp4
